# Model definition and evaluation

This notebook defines, trains, and evaluates a supervised segmentation model for the Raidium 2025 CT scan segmentation challenge. The watershed baseline (mean DICE ≈ 0.0006) from `2_BaselineModel/` is the reference score this model must beat.

The supervised model will learn organ-specific intensity patterns from the annotated training images, allowing it to assign semantically correct labels — something the watershed algorithm fundamentally cannot do.

## Reference: watershed baseline

The watershed baseline from `2_BaselineModel/` scores a mean DICE of **0.0006** on the 200-image balanced validation split. Every model defined in this notebook is evaluated on the same split for a fair comparison.

| Metric | Watershed baseline | This model |
|---|---|---|
| Mean DICE (val, 200 images) | 0.0006 | TBD |
| Lowest-DICE class | Class 1 (0.0000) | TBD |
| Highest-DICE class | Class 24 (0.0084) | TBD |

This table will be updated after training.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

from skimage.segmentation import watershed
from skimage.filters import sobel, rank
from skimage.morphology import disk
from scipy import ndimage as ndi

warnings.filterwarnings("ignore")

# Paths (notebook lives at 3_Model/; data at repo root)
REPO_ROOT = Path("..").resolve()
TRAIN_DIR = REPO_ROOT / "train-images"
TEST_DIR  = REPO_ROOT / "test-images"
LABEL_CSV = REPO_ROOT / "y_train.csv"
BAL_IDX   = REPO_ROOT / "balanced_indices.npy"

NUM_CLASSES = 54

def load_dataset(image_dir):
    files = sorted(Path(image_dir).glob("*.png"), key=lambda f: int(f.stem))
    return np.stack(
        [cv2.imread(str(f), cv2.IMREAD_GRAYSCALE) for f in tqdm(files, desc=Path(image_dir).name)],
        axis=0
    )

print("Loading train images ...")
data_train = load_dataset(TRAIN_DIR)

print("Loading test images ...")
data_test = load_dataset(TEST_DIR)

print("Loading labels (~20 s) ...")
labels_train = pd.read_csv(LABEL_CSV, index_col=0).T   # -> (N_images, N_pixels)

balanced_indices = np.load(BAL_IDX)

# Train / val split (same split as baseline notebook)
val_idx      = balanced_indices[:200]
train_idx    = balanced_indices[200:]

data_val     = data_train[val_idx]
data_model   = data_train[train_idx]
labels_val   = labels_train.iloc[val_idx]
labels_model = labels_train.iloc[train_idx]

print(f"\ndata_model  : {data_model.shape}  (training set for supervised model)")
print(f"data_val    : {data_val.shape}   (validation - same split as baseline)")
print(f"labels_model: {labels_model.shape}")
print(f"labels_val  : {labels_val.shape}")


def dice_image(prediction, ground_truth):
    intersection = np.sum(prediction * ground_truth)
    if np.sum(prediction) == 0 and np.sum(ground_truth) == 0:
        return np.nan
    return 2 * intersection / (np.sum(prediction) + np.sum(ground_truth))


def dice_multiclass(prediction, ground_truth):
    return np.array([
        dice_image(prediction == i, ground_truth == i)
        for i in range(1, NUM_CLASSES + 1)
    ])


def dice_pandas(y_true_df, y_pred_df):
    y_pred = y_pred_df.T
    y_true = y_true_df.T
    scores = []
    for i in range(y_true.values.shape[0]):
        scores.append(dice_multiclass(
            y_true.values[i].ravel(),
            y_pred.values[i].ravel()))
    final    = np.stack(scores)
    cls_dice = np.nanmean(final, axis=0)
    return float(np.nanmean(cls_dice)), cls_dice


print("\nDICE functions defined.")

## Model architecture

[Describe your model architecture here. Explain your choice of algorithm, any preprocessing steps, and how the model addresses the limitations identified in the watershed baseline (e.g. inability to assign semantically correct organ labels to segmented regions).]

## Evaluation plan

The model is evaluated on the 200-image balanced validation split (`val_idx = balanced_indices[:200]`). This is the same split used for the watershed baseline, allowing direct comparison.

**Steps:**
1. Run the trained model on `data_val` to produce pixel-level predictions.
2. Compute `dice_pandas(labels_val, predictions)` to get overall mean DICE and per-class DICE.
3. Compare against the watershed baseline (mean DICE = 0.0006).
4. Report per-class breakdown; focus on Classes 1, 2, 6, 8, 14 (all scoring 0.0000 at baseline).

# Model Definition and Evaluation
## Table of Contents
1. [Model Selection](#model-selection)
2. [Feature Engineering](#feature-engineering)
3. [Hyperparameter Tuning](#hyperparameter-tuning)
4. [Implementation](#implementation)
5. [Evaluation Metrics](#evaluation-metrics)
6. [Comparative Analysis](#comparative-analysis)


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report
# Import models you're considering


## Model Selection

[Discuss the type(s) of models you consider for this task, and justify the selection.]



## Feature Engineering

[Describe any additional feature engineering you've performed beyond what was done for the baseline model.]


In [ ]:
# Load the dataset
# Replace 'your_dataset.csv' with the path to your actual dataset
df = pd.read_csv('your_dataset.csv')

# Perform any feature engineering steps
# Example: df['new_feature'] = df['feature1'] + df['feature2']

# Feature and target variable selection
X = df[['your', 'selected', 'features']]
y = df['target_variable']

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Hyperparameter Tuning

[Discuss any hyperparameter tuning methods you've applied, such as Grid Search or Random Search, and the rationale behind them.]


In [ ]:
# Implement hyperparameter tuning
# Example using GridSearchCV with a DecisionTreeClassifier
# param_grid = {'max_depth': [2, 4, 6, 8]}
# grid_search = GridSearchCV(DecisionTreeClassifier(), param_grid, cv=5)
# grid_search.fit(X_train, y_train)


## Implementation

[Implement the final model(s) you've selected based on the above steps.]


In [ ]:
# Implement the final model(s)
# Example: model = YourChosenModel(best_hyperparameters)
# model.fit(X_train, y_train)


## Evaluation Metrics

[Clearly specify which metrics you'll use to evaluate the model performance, and why you've chosen these metrics.]


In [ ]:
# Evaluate the model using your chosen metrics
# Example for classification
# y_pred = model.predict(X_test)
# print(classification_report(y_test, y_pred))

# Example for regression
# mse = mean_squared_error(y_test, y_pred)

# Your evaluation code here


## Comparative Analysis

[Compare the performance of your model(s) against the baseline model. Discuss any improvements or setbacks and the reasons behind them.]


In [ ]:
# Comparative Analysis code (if applicable)
# Example: comparing accuracy of the baseline model and the new model
# print(f"Baseline Model Accuracy: {baseline_accuracy}, New Model Accuracy: {new_model_accuracy}")
